<a href="https://colab.research.google.com/github/shin584/project/blob/ensemble_system/scanner%2Bensemble%2BUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 모델 다운로드&로드


In [ ]:
!wget -O model.keras "https://github.com/shin584/project/raw/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras"
!wget -O model_sa.keras "https://github.com/shin584/project/raw/ensemble_system/models/SaCas9.keras"
!wget -O model_cas12a.keras "https://github.com/shin584/project/raw/ensemble_system/models/Cas12a_Only.keras"

--2026-05-19 08:47:05--  https://github.com/shin584/project/raw/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shin584/project/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras [following]
--2026-05-19 08:47:05--  https://raw.githubusercontent.com/shin584/project/ensemble_system/models/Multi-Cas_1IN_9Hydra_Divide_Testset.keras
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1029645 (1006K) [application/octet-stream]
Saving to: ‘model.keras’

model.keras         100%[===================>]   1006K  --.-KB/s    in 0.05

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("model.keras")
model_sa = tf.keras.models.load_model("model_sa.keras")
model_cas12a = tf.keras.models.load_model("model_cas12a.keras")

# ngrok 패키지 설치

In [ ]:
!pip install streamlit pyngrok

# pam_scanner.py 파일 저장

In [ ]:
%%writefile /content/pam_scanner.py
"""
pam_scanner.py
──────────────
PAM 서열 탐색 전담 모듈.

포함 내용:
  - CasConfig        : Cas 단백질 설정 (PAM, 가이드 길이, 절단 위치 등)
  - CandidateSite    : 개별 절단 후보 데이터 클래스
  - ScanResult       : 스캔 결과 컨테이너
  - 유틸리티 함수    : iupac_to_regex, reverse_complement, validate_sequence,
                       gaussian_penalty, _wcswidth, _ljust_wide
  - PAM 탐색 함수    : has_spcas9_pam, has_sacas9_pam, has_cas12a_pam,
                       check_all_pams, scan_pam
"""

import re
import math
import unicodedata
from dataclasses import dataclass, field
from typing import Optional


# ════════════════════════════════════════════════
# 섹션 1. 설정
# ════════════════════════════════════════════════

@dataclass(frozen=True)  # 객체 읽기 전용, 해시 가능
class CasConfig:
    """
    Cas 단백질 한 종류의 PAM/절단 규칙 정의.
    새 단백질 추가 시 이 클래스 인스턴스만 하나 더 만들면 됨.
    """
    name:               str   # 유전자 가위 이름
    pam_iupac:          str   # PAM 서열 규칙
    guide_len:          int   # 모델이 인식할 서열의 길이 결정
    pam_position:       str   # '3prime' | '5prime' / PAM 위치
    cut_offset:         int   # DNA가 잘리는 위치
    model_path:         str   # 효율 예측할 때 어떤 모델 파일(.onnx)을 사용할 것인지 정해주는 경로
    model_input_len:    int

# 3가지 유전자 가위의 구체적인 명세서
CAS_CONFIGS: list[CasConfig] = [
    CasConfig(
        name         = "SpCas9",
        pam_iupac    = "NGG",
        guide_len    = 20,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model.keras",
        model_input_len = 30,
    ),
    CasConfig(
        name         = "SaCas9",
        pam_iupac    = "NNGRRT",
        guide_len    = 21,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model_sa.keras",
        model_input_len = 36,
    ),
    CasConfig(
        name         = "Cas12a",
        pam_iupac    = "TTTV",
        guide_len    = 23,
        pam_position = "5prime",
        cut_offset   = 20,
        model_path   = "/content/model_cas12a.keras",
        model_input_len = 34,
    ),
]

# 이후에 추가 모델의 PAM 서열을 인식해야 할 경우를 대비해 모든 IUPAC 넣어놓음
_IUPAC_TABLE: dict[str, str] = {
    'A': 'A', 'T': 'T', 'G': 'G', 'C': 'C',
    'N': '[ATGC]',
    'R': '[AG]',
    'Y': '[CT]',
    'S': '[GC]',
    'W': '[AT]',
    'K': '[GT]',
    'M': '[AC]',
    'B': '[CGT]',
    'D': '[AGT]',
    'H': '[ACT]',
    'V': '[ACG]',
}

# ════════════════════════════════════════════════
# 섹션 2. 데이터 구조
# ════════════════════════════════════════════════

@dataclass
class CandidateSite:
    """유효한 절단 후보 부위 하나의 정보"""
    cas_type:    str            # 가위 종류
    strand:      str            # 가닥 방향(정방향 +, 역방향 -)
    pam_start:   int            # PAM 시작점(인덱스 형태)
    cut_pos:     int            # 절단 위치
    distance:    int            # 변이와의 거리
    guide_seq:   str            # gRNA 서열 (사용자에게 보여줄 가이드 서열, 20/21/23bp)
    pam_seq:     str            # 발견된 PAM 서열
    model_input_seq: str = ""   # AI 모델 입력용 서열(30bp)
    encoded_seq: list = field(default_factory=list)  # 원핫 인코딩 데이터 저장용
    raw_score:   float = 0.0    # 순수 절단 효율 (0~1)
    final_score: float = 0.0    # 거리 패널티 적용 후 %로 변환한 최종 값


@dataclass
class ScanResult:
    """전체 스캔 결과 컨테이너"""
    input_seq:  str                                                         # 사용자가 입력한 81bp DNA 서열
    center_idx: int                        = 40                             # 타겟 변위의 위치
    sites:      list[CandidateSite]        = field(default_factory=list)    # 발견된 모든 CandidateSite들

    def by_cas(self, cas_type: str) -> list[CandidateSite]:
        """필터링 - 특정 가위 종류만 골라냄"""
        return [s for s in self.sites if s.cas_type == cas_type]

    def sorted_by_score(self) -> list[CandidateSite]:
        """정렬 - 모든 후보의 점수를 내림차순으로 정렬"""
        return sorted(self.sites, key=lambda s: s.final_score, reverse=True)

    def __repr__(self) -> str:
        """요약 출력 - 가위별로 찾은 PAM 개수"""
        counts = {cfg.name: len(self.by_cas(cfg.name)) for cfg in CAS_CONFIGS}
        parts  = ", ".join(f"{k}={v}" for k, v in counts.items())
        return f"ScanResult({parts}, total={len(self.sites)})"


# ════════════════════════════════════════════════
# 섹션 3. 유틸리티 함수
# ════════════════════════════════════════════════

def one_hot_encode(seq: str) -> list:
    """DNA 서열을 4xL 형태의 원핫 인코딩으로 변환"""
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'R': [0.5, 0, 0.5, 0],    # A or G
        'Y': [0, 0.5, 0, 0.5],    # C or T
        'S': [0, 0.5, 0.5, 0],    # G or C
        'W': [0.5, 0, 0, 0.5],    # A or T
        'K': [0, 0, 0.5, 0.5],    # G or T
        'M': [0.5, 0.5, 0, 0],    # A or C
        'B': [0, 0.33, 0.33, 0.33],
        'D': [0.33, 0, 0.33, 0.33],
        'H': [0.33, 0.33, 0, 0.33],
        'V': [0.33, 0.33, 0.33, 0],
        'N': [0.25, 0.25, 0.25, 0.25]
    }
    return [mapping.get(b, [0,0,0,0]) for b in seq.upper()]

def iupac_to_regex(pam: str) -> str:
    """IUPAC PAM 서열 → 파이썬 정규표현식 문자열"""
    try:
        return ''.join(_IUPAC_TABLE[b] for b in pam.upper())
    except KeyError as e:
        raise ValueError(f"알 수 없는 IUPAC 코드: {e}")

def reverse_complement(seq: str) -> str:
    """DNA 서열의 역상보(Reverse Complement) 반환 (IUPAC 심볼 포함)"""
    table = str.maketrans("ACGTRYSWKMBDHVNacgtryswkmbdhvn",
                           "TGCAYRSWMKVHDBNtgcayrswmkvhdbn")
    return seq.translate(table)[::-1]


# IUPAC 심볼별 허용 염기 집합
_IUPAC_BASES: dict[str, set[str]] = {
    'A': {'A'}, 'T': {'T'}, 'G': {'G'}, 'C': {'C'},
    'N': {'A','T','G','C'},
    'R': {'A','G'}, 'Y': {'C','T'}, 'S': {'G','C'},
    'W': {'A','T'}, 'K': {'G','T'}, 'M': {'A','C'},
    'B': {'C','G','T'}, 'D': {'A','G','T'},
    'H': {'A','C','T'}, 'V': {'A','C','G'},
}

def iupac_match(seq_char: str, pam_char: str) -> bool:
    """
    입력 서열의 문자 하나와 PAM의 IUPAC 심볼이 호환되는지 확인.
    양쪽 모두 IUPAC일 수 있으므로 허용 염기 집합의 교집합으로 판단.
    예) seq_char='R'(A or G), pam_char='N'(A/T/G/C) → 교집합 {A,G} → True
    """
    return bool(_IUPAC_BASES[seq_char] & _IUPAC_BASES[pam_char])


def iupac_pam_search(seq: str, pam: str) -> list[tuple[int, int, str]]:
    """
    입력 서열에서 PAM 패턴을 IUPAC 호환 방식으로 탐색.
    정규식 대신 위치별로 직접 비교하므로 입력 서열에 IUPAC 심볼이 있어도 동작.

    Returns
    -------
    list of (start, end, matched_window)
    """
    matches = []
    pam_len = len(pam)
    for i in range(len(seq) - pam_len + 1): # 윈도우가 전체 서열을 벗어나지 않도록 반복 범위 지정
        window = seq[i:i + pam_len] # 현재 인덱스에서 PAM 길이만큼 서열 잘라내어 window 변수에 저장
        if all(iupac_match(window[j], pam[j]) for j in range(pam_len)):
            matches.append((i, i + pam_len, window))
    return matches

def validate_sequence(seq: str) -> str:
    """입력 서열 유효성 검사 (81bp 고정, ATGCN 허용)"""
    seq = "".join(seq.upper().split())
    if len(seq) != 81:
        raise ValueError(
            f"입력 서열은 81bp여야 합니다. (현재: {len(seq)}bp)\n"
            "변이 위치 기준 앞뒤 40bp씩 총 81bp를 입력해 주세요."
        )
    VALID_IUPAC = set("ATGCNRYSWKMBDHV")  # 전체 IUPAC 허용
    invalid = set(seq) - VALID_IUPAC
    if invalid:
        raise ValueError(f"허용되지 않는 문자 포함: {invalid}")
    return seq


def gaussian_penalty(distance: int, sigma: float = 10.0) -> float:
    """거리 기반 가우시안 가중치 (거리 멀수록 0 수렴)
    sigma 값 줄이면 감점 폭 커짐"""
    return math.exp(-(distance ** 2) / (2 * sigma ** 2))


def _wcswidth(s: str) -> int:
    """터미널 실제 출력 폭 계산 (한글·전각문자=2, 그 외=1)"""
    return sum(2 if unicodedata.east_asian_width(c) in ('W', 'F') else 1 for c in s)


def _ljust_wide(s: str, width: int) -> str:
    """터미널 폭 기준 왼쪽 정렬 패딩 (한글 포함 문자열도 정확히 맞춤)"""
    return s + ' ' * max(width - _wcswidth(s), 0)


# ════════════════════════════════════════════════
# 섹션 4-1. 개별 PAM 존재 여부 탐색 함수
# ════════════════════════════════════════════════

def has_spcas9_pam(seq: str) -> bool:
    """
    서열 내 SpCas9 PAM(NGG) 존재 여부 확인.
    양쪽 가닥(정방향 NGG / 역방향에서 NGG) 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NGG"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_sacas9_pam(seq: str) -> bool:
    """
    서열 내 SaCas9 PAM(NNGRRT) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NNGRRT"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_cas12a_pam(seq: str) -> bool:
    """
    서열 내 Cas12a PAM(TTTV, V=A/C/G) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("TTTV"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def check_all_pams(seq: str) -> dict[str, bool]:
    """
    세 가지 Cas 단백질의 PAM 존재 여부를 한 번에 반환.

    Returns
    -------
    dict  예: {"SpCas9": True, "SaCas9": False, "Cas12a": True}
    """
    return {
        "SpCas9": has_spcas9_pam(seq),
        "SaCas9": has_sacas9_pam(seq),
        "Cas12a": has_cas12a_pam(seq),
    }


# ════════════════════════════════════════════════
# 섹션 4-2. PAM 탐색 (공통 스캔 로직)
# ════════════════════════════════════════════════

def _calc_cut_pos(cfg: CasConfig, pam_start: int, pam_end: int,
                  strand: str, seq_len: int) -> int:
    """절단 위치(원본 서열 기준) 계산. 역가닥은 좌표 변환."""
    if cfg.pam_position == "3prime":    # SpCas9, SaCas9는 PAM이 가이드 서열 뒤에 존재
        raw_cut = pam_start + cfg.cut_offset
    else:                               # Cas12a는 PAM이 가이드 서열 앞에 존재
        raw_cut = pam_end + cfg.cut_offset
    return raw_cut if strand == '+' else seq_len - raw_cut - 1

def extract_model_input_window(search_seq: str, pam_start: int, pam_end: int,
                                cfg: CasConfig) -> str:
    seq_len      = len(search_seq)
    window_len   = cfg.model_input_len  # ← 고정 30 대신 동적으로

    if cfg.pam_position == "3prime":
        window_start = pam_start - cfg.guide_len - 4
        window_end   = window_start + window_len
    else:
        window_start = pam_start
        window_end   = window_start + window_len

    if window_start < 0 or window_end > seq_len:
        return ""

    return search_seq[window_start:window_end]

def scan_pam(seq: str, cfg: CasConfig,
             center: int = 40, max_dist: int = 15) -> list[CandidateSite]:
    """단일 CasConfig 기준으로 양쪽 가닥에서 PAM을 탐색.
    입력 서열에 IUPAC 심볼이 포함되어 있어도 정상 동작."""
    sites:  list[CandidateSite] = []
    seq_len = len(seq)

    for strand, search_seq in [('+', seq), ('-', reverse_complement(seq))]:
        for pam_start, pam_end, matched_window in iupac_pam_search(search_seq, cfg.pam_iupac):

            if cfg.pam_position == "3prime":            # SpCas9, SaCas9
                guide_start = pam_start - cfg.guide_len
                guide_end   = pam_start
                if guide_start < 0:
                    continue
            else:                                       # Cas12a
                guide_start = pam_end
                guide_end   = pam_end + cfg.guide_len
                if guide_end > seq_len:
                    continue

            cut_pos  = _calc_cut_pos(cfg, pam_start, pam_end, strand, seq_len)
            distance = abs(cut_pos - center)
            if distance > max_dist:
                continue

            model_seq = extract_model_input_window(search_seq, pam_start, pam_end, cfg)  # ← 먼저 추출

            sites.append(CandidateSite(
                cas_type        = cfg.name,
                strand          = strand,
                pam_start       = pam_start,
                cut_pos         = cut_pos,
                distance        = distance,
                guide_seq       = search_seq[guide_start:guide_end],
                pam_seq         = matched_window,
                model_input_seq = model_seq,
                encoded_seq     = one_hot_encode(model_seq) if model_seq else [],
            ))

    return sites


def print_pam_analysis(input_dna: str) -> ScanResult:
    clean_seq = validate_sequence(input_dna)

    # 2. 결과 저장을 위한 컨테이너
    all_results = ScanResult(input_seq=clean_seq)

    # 3. 모든 Cas 설정에 대해 스캔 수행
    print(f"\n🔍 DNA 서열 분석 시작 (길이: {len(clean_seq)}bp)")
    print("-" * 60)

    for cfg in CAS_CONFIGS:
        found_sites = scan_pam(clean_seq, cfg)
        all_results.sites.extend(found_sites)

        # 상세 결과 출력 (터미널)
        print(f"[{cfg.name}] 탐색 결과:")
        if not found_sites:
            print("  - 발견된 PAM 서열이 없습니다.")
        for site in found_sites:
            print(f"  · 가닥: {site.strand} | PAM: {site.pam_seq} | 위치: {site.pam_start} | 가이드: {site.guide_seq}")
        print("-" * 60)

    # 4. 요약 표 출력
    print("\n📊 [PAM 탐색 최종 요약]")
    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'Cas 모델':^13} | {'발견된 개수':^11} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")

    for cfg in CAS_CONFIGS:
        count = len(all_results.by_cas(cfg.name))
        # 모델명과 개수를 표 형식에 맞춰 출력
        print(f"| {cfg.name:<13} | {count:^13} |")

    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'합계':<13} | {len(all_results.sites):^13} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")

    return all_results

import json
import numpy as np
import tensorflow as tf

targets = [
    'SpCas9', 'SpCas9-NG', 'VRQR variant', 'xCas',
    'Sniper-Cas9', 'SpCas9-HF.1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
]

def one_hot_encode_dna(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    return np.array([mapping.get(b, [0,0,0,0]) for b in sequence.upper()])



Overwriting /content/pam_scanner.py


# predict.py

In [ ]:
%%writefile /content/predict.py
import numpy as np
import tensorflow as tf
from pam_scanner import gaussian_penalty

MODEL_SCORE_SCALE = {
    'SpCas9': 100,
    'SaCas9': 100,
    'Cas12a': 1,    # 이미 0~100 스케일
}

def load_models():
    model        = tf.keras.models.load_model("/content/model.keras")
    model_sa     = tf.keras.models.load_model("/content/model_sa.keras")
    model_cas12a = tf.keras.models.load_model("/content/model_cas12a.keras")
    return model, model_sa, model_cas12a

def one_hot_encode_dna(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    return np.array([mapping.get(b, [0,0,0,0]) for b in sequence.upper()])

def run_prediction(all_results, model, model_sa, model_cas12a):
    for site in all_results.sites:
        if not site.model_input_seq:
            continue
        X = np.expand_dims(one_hot_encode_dna(site.model_input_seq), axis=0)

        if site.cas_type == 'SpCas9':
            raw = float(model.predict(X, verbose=0)[0][0][0])
        elif site.cas_type == 'SaCas9':
            raw = float(model_sa.predict(X, verbose=0)[0][0])
        elif site.cas_type == 'Cas12a':
            raw = float(model_cas12a.predict(X, verbose=0)[0][0])
        else:
            raw = 0.0

        scale            = MODEL_SCORE_SCALE.get(site.cas_type, 100)
        penalty          = gaussian_penalty(site.distance)
        site.raw_score   = raw
        site.final_score = raw * penalty * 100

    return all_results

Overwriting /content/predict.py


# app.py 파일 저장

In [ ]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
from pam_scanner import print_pam_analysis, validate_sequence, CAS_CONFIGS
from predict import load_models, run_prediction

st.set_page_config(page_title="CRISPR PAM Scanner", layout="wide")
st.title("🧬 CRISPR PAM Scanner")
st.markdown("81bp DNA 서열을 입력하면 PAM 탐색 및 절단 효율을 예측합니다.")

@st.cache_resource
def get_models():
    return load_models()

model, model_sa, model_cas12a = get_models()

dna_input = st.text_input("81bp DNA 서열 입력", placeholder="ATCG...")
run_btn   = st.button("🔍 분석 시작")

if run_btn and dna_input:
    try:
        validate_sequence(dna_input)

        with st.spinner("PAM 탐색 중..."):
            all_results = print_pam_analysis(dna_input)

        st.subheader("📊 PAM 탐색 결과")
        for cfg in CAS_CONFIGS:
            sites = all_results.by_cas(cfg.name)
            st.markdown(f"**{cfg.name}**: {len(sites)}개 발견")

        with st.spinner("모델 예측 중..."):
            all_results = run_prediction(all_results, model, model_sa, model_cas12a)

        st.subheader("🤖 절단 효율 예측 결과")
        rows = []
        for i, site in enumerate(all_results.sorted_by_score(), start=1):
            rows.append({
                '순위':        i,
                'Cas 종류':    site.cas_type,
                'PAM':         site.pam_seq,
                '가이드 서열':  site.guide_seq,
                '거리':        site.distance,
                'Raw Score':   round(site.raw_score, 4),
                '최종 효율(%)': round(site.final_score, 2),
            })
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

    except ValueError as e:
        st.error(f"❌ 오류: {e}")

Overwriting /content/app.py


# Streamlit 실행

In [ ]:
from pyngrok import ngrok
import subprocess

ngrok.set_auth_token("3DIe6sUE6TpnmZzlxT7FH6DvWJa_5dt2N3s7W2mwVGfT2ahZH")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501",
                            "--server.headless=true"])

public_url = ngrok.connect(8501)
print(f"✅ 접속 URL: {public_url}")

✅ 접속 URL: NgrokTunnel: "https://emcee-slip-grit.ngrok-free.dev" -> "http://localhost:8501"


# pam_scanner.py


In [ ]:
"""
pam_scanner.py
──────────────
PAM 서열 탐색 전담 모듈.

포함 내용:
  - CasConfig        : Cas 단백질 설정 (PAM, 가이드 길이, 절단 위치 등)
  - CandidateSite    : 개별 절단 후보 데이터 클래스
  - ScanResult       : 스캔 결과 컨테이너
  - 유틸리티 함수    : iupac_to_regex, reverse_complement, validate_sequence,
                       gaussian_penalty, _wcswidth, _ljust_wide
  - PAM 탐색 함수    : has_spcas9_pam, has_sacas9_pam, has_cas12a_pam,
                       check_all_pams, scan_pam
"""

import re
import math
import unicodedata
from dataclasses import dataclass, field
from typing import Optional


# ════════════════════════════════════════════════
# 섹션 1. 설정
# ════════════════════════════════════════════════

@dataclass(frozen=True)  # 객체 읽기 전용, 해시 가능
class CasConfig:
    """
    Cas 단백질 한 종류의 PAM/절단 규칙 정의.
    새 단백질 추가 시 이 클래스 인스턴스만 하나 더 만들면 됨.
    """
    name:               str   # 유전자 가위 이름
    pam_iupac:          str   # PAM 서열 규칙
    guide_len:          int   # 모델이 인식할 서열의 길이 결정
    pam_position:       str   # '3prime' | '5prime' / PAM 위치
    cut_offset:         int   # DNA가 잘리는 위치
    model_path:         str   # 효율 예측할 때 어떤 모델 파일(.onnx)을 사용할 것인지 정해주는 경로
    model_input_len:    int

# 3가지 유전자 가위의 구체적인 명세서
CAS_CONFIGS: list[CasConfig] = [
    CasConfig(
        name         = "SpCas9",
        pam_iupac    = "NGG",
        guide_len    = 20,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model.keras",
        model_input_len = 30,
    ),
    CasConfig(
        name         = "SaCas9",
        pam_iupac    = "NNGRRT",
        guide_len    = 21,
        pam_position = "3prime",
        cut_offset   = -3,
        model_path   = "/content/model_sa.keras",
        model_input_len = 36,
    ),
    CasConfig(
        name         = "Cas12a",
        pam_iupac    = "TTTV",
        guide_len    = 23,
        pam_position = "5prime",
        cut_offset   = 20,
        model_path   = "/content/model_cas12a.keras",
        model_input_len = 34,
    ),
]

# 이후에 추가 모델의 PAM 서열을 인식해야 할 경우를 대비해 모든 IUPAC 넣어놓음
_IUPAC_TABLE: dict[str, str] = {
    'A': 'A', 'T': 'T', 'G': 'G', 'C': 'C',
    'N': '[ATGC]',
    'R': '[AG]',
    'Y': '[CT]',
    'S': '[GC]',
    'W': '[AT]',
    'K': '[GT]',
    'M': '[AC]',
    'B': '[CGT]',
    'D': '[AGT]',
    'H': '[ACT]',
    'V': '[ACG]',
}

# ════════════════════════════════════════════════
# 섹션 2. 데이터 구조
# ════════════════════════════════════════════════

@dataclass
class CandidateSite:
    """유효한 절단 후보 부위 하나의 정보"""
    cas_type:    str            # 가위 종류
    strand:      str            # 가닥 방향(정방향 +, 역방향 -)
    pam_start:   int            # PAM 시작점(인덱스 형태)
    cut_pos:     int            # 절단 위치
    distance:    int            # 변이와의 거리
    guide_seq:   str            # gRNA 서열 (사용자에게 보여줄 가이드 서열, 20/21/23bp)
    pam_seq:     str            # 발견된 PAM 서열
    model_input_seq: str = ""   # AI 모델 입력용 서열(30bp)
    encoded_seq: list = field(default_factory=list)  # 원핫 인코딩 데이터 저장용
    raw_score:   float = 0.0    # 순수 절단 효율 (0~1)
    final_score: float = 0.0    # 거리 패널티 적용 후 %로 변환한 최종 값


@dataclass
class ScanResult:
    """전체 스캔 결과 컨테이너"""
    input_seq:  str                                                         # 사용자가 입력한 81bp DNA 서열
    center_idx: int                        = 40                             # 타겟 변위의 위치
    sites:      list[CandidateSite]        = field(default_factory=list)    # 발견된 모든 CandidateSite들

    def by_cas(self, cas_type: str) -> list[CandidateSite]:
        """필터링 - 특정 가위 종류만 골라냄"""
        return [s for s in self.sites if s.cas_type == cas_type]

    def sorted_by_score(self) -> list[CandidateSite]:
        """정렬 - 모든 후보의 점수를 내림차순으로 정렬"""
        return sorted(self.sites, key=lambda s: s.final_score, reverse=True)

    def __repr__(self) -> str:
        """요약 출력 - 가위별로 찾은 PAM 개수"""
        counts = {cfg.name: len(self.by_cas(cfg.name)) for cfg in CAS_CONFIGS}
        parts  = ", ".join(f"{k}={v}" for k, v in counts.items())
        return f"ScanResult({parts}, total={len(self.sites)})"


# ════════════════════════════════════════════════
# 섹션 3. 유틸리티 함수
# ════════════════════════════════════════════════

def one_hot_encode(seq: str) -> list:
    """DNA 서열을 4xL 형태의 원핫 인코딩으로 변환"""
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'R': [0.5, 0, 0.5, 0],    # A or G
        'Y': [0, 0.5, 0, 0.5],    # C or T
        'S': [0, 0.5, 0.5, 0],    # G or C
        'W': [0.5, 0, 0, 0.5],    # A or T
        'K': [0, 0, 0.5, 0.5],    # G or T
        'M': [0.5, 0.5, 0, 0],    # A or C
        'B': [0, 0.33, 0.33, 0.33],
        'D': [0.33, 0, 0.33, 0.33],
        'H': [0.33, 0.33, 0, 0.33],
        'V': [0.33, 0.33, 0.33, 0],
        'N': [0.25, 0.25, 0.25, 0.25]
    }
    return [mapping.get(b, [0,0,0,0]) for b in seq.upper()]

def iupac_to_regex(pam: str) -> str:
    """IUPAC PAM 서열 → 파이썬 정규표현식 문자열"""
    try:
        return ''.join(_IUPAC_TABLE[b] for b in pam.upper())
    except KeyError as e:
        raise ValueError(f"알 수 없는 IUPAC 코드: {e}")

def reverse_complement(seq: str) -> str:
    """DNA 서열의 역상보(Reverse Complement) 반환 (IUPAC 심볼 포함)"""
    table = str.maketrans("ACGTRYSWKMBDHVNacgtryswkmbdhvn",
                           "TGCAYRSWMKVHDBNtgcayrswmkvhdbn")
    return seq.translate(table)[::-1]


# IUPAC 심볼별 허용 염기 집합
_IUPAC_BASES: dict[str, set[str]] = {
    'A': {'A'}, 'T': {'T'}, 'G': {'G'}, 'C': {'C'},
    'N': {'A','T','G','C'},
    'R': {'A','G'}, 'Y': {'C','T'}, 'S': {'G','C'},
    'W': {'A','T'}, 'K': {'G','T'}, 'M': {'A','C'},
    'B': {'C','G','T'}, 'D': {'A','G','T'},
    'H': {'A','C','T'}, 'V': {'A','C','G'},
}

def iupac_match(seq_char: str, pam_char: str) -> bool:
    """
    입력 서열의 문자 하나와 PAM의 IUPAC 심볼이 호환되는지 확인.
    양쪽 모두 IUPAC일 수 있으므로 허용 염기 집합의 교집합으로 판단.
    예) seq_char='R'(A or G), pam_char='N'(A/T/G/C) → 교집합 {A,G} → True
    """
    return bool(_IUPAC_BASES[seq_char] & _IUPAC_BASES[pam_char])


def iupac_pam_search(seq: str, pam: str) -> list[tuple[int, int, str]]:
    """
    입력 서열에서 PAM 패턴을 IUPAC 호환 방식으로 탐색.
    정규식 대신 위치별로 직접 비교하므로 입력 서열에 IUPAC 심볼이 있어도 동작.

    Returns
    -------
    list of (start, end, matched_window)
    """
    matches = []
    pam_len = len(pam)
    for i in range(len(seq) - pam_len + 1): # 윈도우가 전체 서열을 벗어나지 않도록 반복 범위 지정
        window = seq[i:i + pam_len] # 현재 인덱스에서 PAM 길이만큼 서열 잘라내어 window 변수에 저장
        if all(iupac_match(window[j], pam[j]) for j in range(pam_len)):
            matches.append((i, i + pam_len, window))
    return matches

def validate_sequence(seq: str) -> str:
    """입력 서열 유효성 검사 (81bp 고정, ATGCN 허용)"""
    seq = "".join(seq.upper().split())
    if len(seq) != 81:
        raise ValueError(
            f"입력 서열은 81bp여야 합니다. (현재: {len(seq)}bp)\n"
            "변이 위치 기준 앞뒤 40bp씩 총 81bp를 입력해 주세요."
        )
    VALID_IUPAC = set("ATGCNRYSWKMBDHV")  # 전체 IUPAC 허용
    invalid = set(seq) - VALID_IUPAC
    if invalid:
        raise ValueError(f"허용되지 않는 문자 포함: {invalid}")
    return seq


def gaussian_penalty(distance: int, sigma: float = 10.0) -> float:
    """거리 기반 가우시안 가중치 (거리 멀수록 0 수렴)
    sigma 값 줄이면 감점 폭 커짐"""
    return math.exp(-(distance ** 2) / (2 * sigma ** 2))


def _wcswidth(s: str) -> int:
    """터미널 실제 출력 폭 계산 (한글·전각문자=2, 그 외=1)"""
    return sum(2 if unicodedata.east_asian_width(c) in ('W', 'F') else 1 for c in s)


def _ljust_wide(s: str, width: int) -> str:
    """터미널 폭 기준 왼쪽 정렬 패딩 (한글 포함 문자열도 정확히 맞춤)"""
    return s + ' ' * max(width - _wcswidth(s), 0)


# ════════════════════════════════════════════════
# 섹션 4-1. 개별 PAM 존재 여부 탐색 함수
# ════════════════════════════════════════════════

def has_spcas9_pam(seq: str) -> bool:
    """
    서열 내 SpCas9 PAM(NGG) 존재 여부 확인.
    양쪽 가닥(정방향 NGG / 역방향에서 NGG) 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NGG"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_sacas9_pam(seq: str) -> bool:
    """
    서열 내 SaCas9 PAM(NNGRRT) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("NNGRRT"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def has_cas12a_pam(seq: str) -> bool:
    """
    서열 내 Cas12a PAM(TTTV, V=A/C/G) 존재 여부 확인.
    양쪽 가닥 모두 탐색.
    """
    seq    = seq.upper()
    pam_re = re.compile(iupac_to_regex("TTTV"))
    return bool(pam_re.search(seq)) or bool(pam_re.search(reverse_complement(seq)))


def check_all_pams(seq: str) -> dict[str, bool]:
    """
    세 가지 Cas 단백질의 PAM 존재 여부를 한 번에 반환.

    Returns
    -------
    dict  예: {"SpCas9": True, "SaCas9": False, "Cas12a": True}
    """
    return {
        "SpCas9": has_spcas9_pam(seq),
        "SaCas9": has_sacas9_pam(seq),
        "Cas12a": has_cas12a_pam(seq),
    }


# ════════════════════════════════════════════════
# 섹션 4-2. PAM 탐색 (공통 스캔 로직)
# ════════════════════════════════════════════════

def _calc_cut_pos(cfg: CasConfig, pam_start: int, pam_end: int,
                  strand: str, seq_len: int) -> int:
    """절단 위치(원본 서열 기준) 계산. 역가닥은 좌표 변환."""
    if cfg.pam_position == "3prime":    # SpCas9, SaCas9는 PAM이 가이드 서열 뒤에 존재
        raw_cut = pam_start + cfg.cut_offset
    else:                               # Cas12a는 PAM이 가이드 서열 앞에 존재
        raw_cut = pam_end + cfg.cut_offset
    return raw_cut if strand == '+' else seq_len - raw_cut - 1

def extract_model_input_window(search_seq: str, pam_start: int, pam_end: int,
                                cfg: CasConfig) -> str:
    seq_len      = len(search_seq)
    window_len   = cfg.model_input_len  # ← 고정 30 대신 동적으로

    if cfg.pam_position == "3prime":
        window_start = pam_start - cfg.guide_len - 4
        window_end   = window_start + window_len
    else:
        window_start = pam_start
        window_end   = window_start + window_len

    if window_start < 0 or window_end > seq_len:
        return ""

    return search_seq[window_start:window_end]

def scan_pam(seq: str, cfg: CasConfig,
             center: int = 40, max_dist: int = 15) -> list[CandidateSite]:
    """단일 CasConfig 기준으로 양쪽 가닥에서 PAM을 탐색.
    입력 서열에 IUPAC 심볼이 포함되어 있어도 정상 동작."""
    sites:  list[CandidateSite] = []
    seq_len = len(seq)

    for strand, search_seq in [('+', seq), ('-', reverse_complement(seq))]:
        for pam_start, pam_end, matched_window in iupac_pam_search(search_seq, cfg.pam_iupac):

            if cfg.pam_position == "3prime":            # SpCas9, SaCas9
                guide_start = pam_start - cfg.guide_len
                guide_end   = pam_start
                if guide_start < 0:
                    continue
            else:                                       # Cas12a
                guide_start = pam_end
                guide_end   = pam_end + cfg.guide_len
                if guide_end > seq_len:
                    continue

            cut_pos  = _calc_cut_pos(cfg, pam_start, pam_end, strand, seq_len)
            distance = abs(cut_pos - center)
            if distance > max_dist:
                continue

            model_seq = extract_model_input_window(search_seq, pam_start, pam_end, cfg)  # ← 먼저 추출

            sites.append(CandidateSite(
                cas_type        = cfg.name,
                strand          = strand,
                pam_start       = pam_start,
                cut_pos         = cut_pos,
                distance        = distance,
                guide_seq       = search_seq[guide_start:guide_end],
                pam_seq         = matched_window,
                model_input_seq = model_seq,
                encoded_seq     = one_hot_encode(model_seq) if model_seq else [],
            ))

    return sites


def print_pam_analysis(input_dna: str) -> ScanResult:
    clean_seq = validate_sequence(input_dna)

    # 2. 결과 저장을 위한 컨테이너
    all_results = ScanResult(input_seq=clean_seq)

    # 3. 모든 Cas 설정에 대해 스캔 수행
    print(f"\n🔍 DNA 서열 분석 시작 (길이: {len(clean_seq)}bp)")
    print("-" * 60)

    for cfg in CAS_CONFIGS:
        found_sites = scan_pam(clean_seq, cfg)
        all_results.sites.extend(found_sites)

        # 상세 결과 출력 (터미널)
        print(f"[{cfg.name}] 탐색 결과:")
        if not found_sites:
            print("  - 발견된 PAM 서열이 없습니다.")
        for site in found_sites:
            print(f"  · 가닥: {site.strand} | PAM: {site.pam_seq} | 위치: {site.pam_start} | 가이드: {site.guide_seq}")
        print("-" * 60)

    # 4. 요약 표 출력
    print("\n📊 [PAM 탐색 최종 요약]")
    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'Cas 모델':^13} | {'발견된 개수':^11} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")

    for cfg in CAS_CONFIGS:
        count = len(all_results.by_cas(cfg.name))
        # 모델명과 개수를 표 형식에 맞춰 출력
        print(f"| {cfg.name:<13} | {count:^13} |")

    print("+" + "-"*15 + "+" + "-"*15 + "+")
    print(f"| {'합계':<13} | {len(all_results.sites):^13} |")
    print("+" + "-"*15 + "+" + "-"*15 + "+")

    return all_results

import json
import numpy as np
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/project_CAS'
output_path = '/content/scanner_output.json'
model_path = '/content/model.keras'

# 모델 로드 (한 번만 로드)
model = tf.keras.models.load_model(model_path)
print("✅ 모델 로드 완료!")

targets = [
    'SpCas9', 'SpCas9-NG', 'VRQR variant', 'xCas',
    'Sniper-Cas9', 'SpCas9-HF.1', 'eSpCas9(1.1)', 'HypaCas9', 'evoCas9'
]

def one_hot_encode_dna(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    return np.array([mapping.get(b, [0,0,0,0]) for b in sequence.upper()])



# 실행 코드

In [ ]:
if __name__ == "__main__":
    # Cas 종류별로 모델 출력 인덱스 매핑
    CAS_OUTPUT_INDEX = {
        'SpCas9':       0,
        'SaCas9':       None,  # 모델에 SaCas9 출력 없음 → 별도 처리 필요
        'Cas12a':       None,  # 모델에 Cas12a 출력 없음 → 별도 처리 필요
    }

    # SpCas9 변종 전체 인덱스
    SPCAS9_VARIANT_INDEX = {
        'SpCas9':        0,
        'SpCas9-NG':     1,
        'VRQR variant':  2,
        'xCas':          3,
        'Sniper-Cas9':   4,
        'SpCas9-HF.1':   5,
        'eSpCas9(1.1)':  6,
        'HypaCas9':      7,
        'evoCas9':       8,
    }
    while True:
        user_input = input("\n81bp DNA 서열을 입력하세요 (종료: q): ").strip()
        if user_input.lower() == 'q':
            print("프로그램 종료")
            break
        try:
            # 1. 스캐너 실행
            all_results = print_pam_analysis(user_input)

            # 2. 스캐너 결과 저장
            scanner_results = []
            for site in all_results.sites:
                scanner_results.append({
                    'cas_type':       site.cas_type,
                    'guide_seq':      site.guide_seq,
                    'pam_seq':        site.pam_seq,
                    'model_input_seq': site.model_input_seq,  # 추가
                    'distance':       site.distance,           # 가우시안용 추가
                })

            with open(output_path, 'w') as f:
                json.dump(scanner_results, f)

            # 3. 모델 예측 + 가우시안 패널티 적용
            print("\n🤖 모델 예측 중...")
            for site in all_results.sites:
                if not site.model_input_seq:
                    continue

                X = np.expand_dims(one_hot_encode_dna(site.model_input_seq), axis=0)
                preds = model.predict(X, verbose=0)

                # Cas 종류에 따라 해당 출력 인덱스 선택
                if site.cas_type == 'SpCas9':
                    raw = float(model.predict(X, verbose=0)[0][0][0])
                elif site.cas_type == 'SaCas9':
                    raw = float(model_sa.predict(X, verbose=0)[0][0])
                elif site.cas_type == 'Cas12a':
                    raw = float(model_cas12a.predict(X, verbose=0)[0][0])
                else:
                    raw = 0.0

                # 가우시안 패널티 적용 후 site에 저장
                penalty          = gaussian_penalty(site.distance)
                site.raw_score   = raw
                site.final_score = raw * penalty * 100

            # 4. 결과 출력
            print("\n📊 최종 예측 결과 (가우시안 패널티 적용)")
            for site in all_results.sorted_by_score():
                bar = '█' * int(site.final_score // 5)
                print(f"\n📍 Cas: {site.cas_type} | PAM: {site.pam_seq} | Guide: {site.guide_seq}")
                print(f"  raw: {site.raw_score:.4f} | 거리: {site.distance} | 최종: {site.final_score:.2f}%  {bar}")

        except ValueError as e:
            print(f"❌ 오류: {e}")
            print("다시 입력하세요.")


81bp DNA 서열을 입력하세요 (종료: q): TTTGAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCGGAGTCAGTCAGTCAGNNGRRTCAGTCAGTCAGTCAGTCAA

🔍 DNA 서열 분석 시작 (길이: 81bp)
------------------------------------------------------------
[SpCas9] 탐색 결과:
  · 가닥: + | PAM: CGG | 위치: 39 | 가이드: CAGTCAGTCAGTCAGTCAGT
  · 가닥: + | PAM: AGN | 위치: 54 | 가이드: TCAGTCGGAGTCAGTCAGTC
  · 가닥: + | PAM: GNN | 위치: 55 | 가이드: CAGTCGGAGTCAGTCAGTCA
  · 가닥: + | PAM: NNG | 위치: 56 | 가이드: AGTCGGAGTCAGTCAGTCAG
  · 가닥: + | PAM: NGR | 위치: 57 | 가이드: GTCGGAGTCAGTCAGTCAGN
  · 가닥: + | PAM: GRR | 위치: 58 | 가이드: TCGGAGTCAGTCAGTCAGNN
------------------------------------------------------------
[SaCas9] 탐색 결과:
  · 가닥: + | PAM: CGGAGT | 위치: 39 | 가이드: TCAGTCAGTCAGTCAGTCAGT
  · 가닥: + | PAM: NNGRRT | 위치: 56 | 가이드: CAGTCGGAGTCAGTCAGTCAG
------------------------------------------------------------
[Cas12a] 탐색 결과:
  - 발견된 PAM 서열이 없습니다.
------------------------------------------------------------

📊 [PAM 탐색 최종 요약]
+---------------+---------------+
|    Cas 모델     |   발견된 